# EDA Part 3: Segmentation Results

**Author:** Shreya Aggarwal  
**Date:** 2026-02-22  
**GitHub:** https://github.com/shreya1297/Thesis

---

## What is this notebook?

**Segmentation** means teaching the computer to colour in each individual cell
so it can count them, measure their shapes, and compare them.

Think of it like this: you have a photo of a crowd of people. Segmentation is the
process of drawing an outline around each person, giving each person their own label,
and then measuring their height, width, etc.

We used **Cellpose** — a deep learning model — to segment endothelial cells.
We tested **two models**:
- **cpsam** (Transformer): a powerful modern model, ~1.15 GB — like a very smart but slow brain
- **cyto3** (CNN): a classic efficient model, ~25 MB — like a fast experienced brain

And **5 configurations** total:
1. cpsam, Actin channel only (diameter=150)
2. cyto3, Actin only (diameter=150)
3. cyto3, Actin only (diameter=200)
4. cyto3, Actin + VE-cadherin (diameter=150)
5. cyto3, Actin + VE-cadherin (diameter=200)

## Connection to Research Questions

- **SQ1**: How accurately can cells be segmented? → We evaluate segmentation quality (coverage, cell count)
- **SQ2**: Can we extract CPM-relevant morphological features? → We look at area, circularity, eccentricity
- **SQ3** (first look): Do Q vs TNF cells differ in shape? → We compare grouped distributions

---

## Glossary

| Term | Plain English explanation |
|------|---------------------------|
| **Coverage** | What fraction of the image area has detected cells. 95% = nearly the whole image is cells |
| **Area** | Size of one cell in pixels. Cells are ~5,000–60,000 pixels depending on size |
| **Circularity** | How round the cell is. 1.0 = perfect circle. <0.7 = elongated/irregular |
| **Eccentricity** | How stretched the cell is. 0 = circle. 1 = straight line. |
| **Solidity** | How compact the cell is. 1 = no concavities (smooth border). <0.9 = jagged border |
| **CPM parameter** | A number in the Cellular Potts Model that controls cell behaviour in simulation |


In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})

# ── Find project root ─────────────────────────────────────────────────────
_p = Path().resolve()
PROJECT_ROOT = None
for _parent in [_p] + list(_p.parents):
    if (_parent / 'Quantification_EndothelialCells_CPM').is_dir():
        PROJECT_ROOT = _parent
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError('Cannot find project root. Run from inside implementation/ folder.')

THESIS_DIR = PROJECT_ROOT / 'Quantification_EndothelialCells_CPM'
OUTPUT_DIR = THESIS_DIR / 'output'
EDA_DIR    = OUTPUT_DIR / 'eda' / 'eda_03'
EDA_DIR.mkdir(parents=True, exist_ok=True)

# ── Load batch summary CSVs for all 5 configs ────────────────────────────
CONFIGS = {
    'cpsam (Actin d=150)':       'batch_t0001_cpsam_tiff',
    'cyto3 (Actin d=150)':       'batch_t0001_cyto3_actin_d150_tiff',
    'cyto3 (Actin d=200)':       'batch_t0001_cyto3_actin_d200_tiff',
    'cyto3 (Multi d=150)':       'batch_t0001_cyto3_multichannel_d150_tiff',
    'cyto3 (Multi d=200)':       'batch_t0001_cyto3_multichannel_d200_tiff',
}

summaries = {}
for label, folder in CONFIGS.items():
    csv_path = OUTPUT_DIR / folder / 'batch_summary.csv'
    if csv_path.exists():
        summaries[label] = pd.read_csv(csv_path)
        print(f'  Loaded {label}: {len(summaries[label])} samples')
    else:
        print(f'  MISSING: {csv_path}')

# Primary config: cpsam
df = summaries['cpsam (Actin d=150)'].copy()
print(f'\nPrimary config (cpsam): {len(df)} samples')

# ── Load per-cell metrics for cpsam ─────────────────────────────────────
cells_dir = OUTPUT_DIR / 'batch_t0001_cpsam_tiff' / 'samples'
meta_dict = {row['sample_name']: row for _, row in df.iterrows()}

cell_dfs = []
for csv_path in sorted(cells_dir.glob('*_metrics.csv')):
    sample_name = csv_path.stem.replace('_metrics', '')
    cell_df = pd.read_csv(csv_path)
    cell_df['sample_name'] = sample_name
    if sample_name in meta_dict:
        cell_df['protein']   = meta_dict[sample_name]['protein']
        cell_df['condition'] = meta_dict[sample_name]['condition']
    cell_dfs.append(cell_df)

cells = pd.concat(cell_dfs, ignore_index=True) if cell_dfs else pd.DataFrame()
print(f'Total detected cells (cpsam, all samples, T0001): {len(cells):,}')
print(f'Columns: {list(cells.columns)}')


## 1. Baseline Model: Classical Thresholding vs Deep Learning

Before evaluating Cellpose, we establish a **baseline** using the simplest classical
segmentation approach: **Otsu thresholding**.

**What Otsu thresholding does (for complete beginners):**  
Imagine the image histogram as two hills — one small dark hill (background) and one
bright hill (cell bodies). Otsu finds the valley between them and draws a line there.
Every pixel above the line = cell. Every pixel below = background.  
No learning. No parameters to tune. Just statistics on pixel intensities.

**Why this is the right baseline:**
- It's the standard classical approach used in cell biology before deep learning
- It tells us: how much does deep learning actually add over a simple threshold?
- If Otsu already works perfectly, Cellpose is unnecessary — we need to show it doesn't

**Limitation:** Otsu labels pixels as foreground/background but **cannot separate
touching cells** — adjacent cells merge into one blob. Cellpose explicitly models
cell boundaries, which is why it produces individual cell outlines rather than blobs.


In [ ]:

from skimage.filters import threshold_otsu
from skimage.measure import label as sk_label, regionprops
from PIL import Image as _PIL_bl

_DATA_BL = PROJECT_ROOT / 'data' / 'JK2513_A1R_HUVEC_dataset cellular pots model'

# Ensure coverage_pct exists (computed formally in Section 3, needed here early)
if 'coverage_pct' not in df.columns:
    df['coverage_pct'] = df['coverage'] * 100

def _find_t0001_bl(sample_name, data_dir):
    for ch3_dir in data_dir.rglob('*_Channel3'):
        if 'Out of focus' in str(ch3_dir):
            continue
        dir_s = ch3_dir.name.replace('_Channel3', '')
        if dir_s == sample_name or sample_name.endswith(dir_s):
            hits = sorted(ch3_dir.glob('*T0001*.tif'))
            if hits:
                return hits[0]
    return None

# ── Select 5 representative samples for baseline comparison ──────────────
_bl_names = (
    df.nlargest(2, 'coverage_pct')['sample_name'].tolist() +    # 2 best cpsam
    df[df['condition'] == 'TNF']['sample_name'].tolist()[:2] +  # 2 TNF samples
    df.nsmallest(1, 'coverage_pct')['sample_name'].tolist()     # 1 worst outlier
)
_bl_names = list(dict.fromkeys(_bl_names))[:5]  # deduplicate

# ── Run Otsu threshold on each ────────────────────────────────────────────
bl_rows = []
for sname in _bl_names:
    actin_p = _find_t0001_bl(sname, _DATA_BL)
    if actin_p is None or not actin_p.exists():
        continue
    img = np.array(_PIL_bl.open(actin_p)).astype(float)

    thresh  = threshold_otsu(img)
    binary  = img > thresh
    cov_otsu = binary.mean() * 100

    # Count connected components (proxy for cells — very rough)
    labeled  = sk_label(binary)
    n_otsu   = sum(1 for r in regionprops(labeled) if r.area > 500)

    cpsam_row   = df[df['sample_name'] == sname]
    cov_cpsam   = float(cpsam_row['coverage_pct'].values[0]) if len(cpsam_row) else None
    cells_cpsam = int(cpsam_row['n_cells'].values[0])        if len(cpsam_row) else None

    short = sname.split('siractin_')[-1] if 'siractin_' in sname else sname[-20:]
    bl_rows.append({
        'Sample': short,
        'Condition': cpsam_row['condition'].values[0] if len(cpsam_row) else '?',
        'Otsu thresh': int(thresh),
        'Otsu coverage %': round(cov_otsu, 1),
        'Otsu components': n_otsu,
        'cpsam coverage %': round(cov_cpsam, 1) if cov_cpsam else None,
        'cpsam cells': cells_cpsam,
    })

bl_df = pd.DataFrame(bl_rows)
print('Baseline (Otsu thresholding) vs Our Model (cpsam):')
print(bl_df.to_string(index=False))
print()
print(f'Otsu  median coverage  : {bl_df["Otsu coverage %"].median():.1f}%')
print(f'cpsam median coverage  : {bl_df["cpsam coverage %"].median():.1f}%')
delta = bl_df["cpsam coverage %"].median() - bl_df["Otsu coverage %"].median()
print(f'Deep learning gain     : +{delta:.1f} percentage points')
print()
print('Key limitation of Otsu: "components" ≠ real cells.')
print('Otsu cannot separate touching cells — adjacent cells merge into one blob.')
print('cpsam explicitly models cell boundaries and separates touching cells correctly.')

# ── Visual comparison for the top-coverage sample ─────────────────────────
_ex_name = _bl_names[0]
_ex_path = _find_t0001_bl(_ex_name, _DATA_BL)
_masks_bl = OUTPUT_DIR / 'batch_t0001_cpsam_tiff' / 'samples'

if _ex_path and _ex_path.exists():
    _img = np.array(_PIL_bl.open(_ex_path)).astype(float)
    _thresh = threshold_otsu(_img)
    _binary = _img > _thresh

    _lo, _hi = _img.min(), np.percentile(_img, 99)
    _disp = np.clip((_img - _lo) / (_hi - _lo), 0, 1)

    fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

    axes[0].imshow(_disp, cmap='gray')
    axes[0].set_title('Input: Raw Actin (Ch3)', fontweight='bold')
    axes[0].axis('off')

    axes[1].imshow(_binary, cmap='gray')
    axes[1].set_title(
        f'Baseline: Otsu Threshold\ncoverage = {_binary.mean()*100:.1f}%  '
        f'(threshold = {int(_thresh)})',
        fontweight='bold'
    )
    axes[1].axis('off')

    _mask_p = _masks_bl / f'{_ex_name}_masks.png'
    if not _mask_p.exists():
        cands = sorted(_masks_bl.glob(f'*{_ex_name[-12:]}*masks*'))
        _mask_p = cands[0] if cands else None

    if _mask_p and Path(str(_mask_p)).exists():
        axes[2].imshow(np.array(_PIL_bl.open(str(_mask_p))))
    else:
        axes[2].text(0.5, 0.5, 'mask not found', ha='center', va='center',
                     transform=axes[2].transAxes, color='gray')

    _ex_row = df[df['sample_name'] == _ex_name]
    _cov = _ex_row['coverage_pct'].values[0] if len(_ex_row) else '?'
    _nc  = _ex_row['n_cells'].values[0]      if len(_ex_row) else '?'
    axes[2].set_title(
        f'Our model: Cellpose cpsam\ncoverage = {_cov:.1f}%  |  {_nc} cells detected',
        fontweight='bold'
    )
    axes[2].axis('off')

    fig.suptitle(
        'Baseline vs Deep Learning — Segmentation Comparison\n'
        f'Sample: {_ex_name.split("siractin_")[-1] if "siractin_" in _ex_name else _ex_name[-20:]}',
        fontsize=12, fontweight='bold'
    )
    plt.tight_layout()
    plt.savefig(EDA_DIR / 'seg_baseline_comparison.png', bbox_inches='tight', dpi=150)
    plt.show()
    print(f'Saved → {EDA_DIR / "seg_baseline_comparison.png"}')


## 1. Quick Overview — How Well Did Segmentation Work?

Let's first compare all 5 configurations side-by-side.

**Key metrics:**
- **Median cells**: how many cells detected in a typical sample
- **Median coverage**: what fraction of the image is covered by detected cells (higher = better)
- **Std coverage**: how consistent the model is across samples (lower = more reliable)

In [ ]:
rows = []
for label, s_df in summaries.items():
    model = 'cpsam (Transformer)' if 'cpsam' in label else 'cyto3 (CNN)'
    rows.append({
        'Configuration': label,
        'Model': model,
        'Median cells': int(s_df['n_cells'].median()),
        'Median coverage %': round(s_df['coverage'].median() * 100, 1),
        'Mean coverage %': round(s_df['coverage'].mean() * 100, 1),
        'Std coverage %': round(s_df['coverage'].std() * 100, 1),
    })

overview = pd.DataFrame(rows)
print('Segmentation model comparison (all 23 samples, T0001):')
print(overview.to_string(index=False))
print('\nKey insight: cpsam achieves highest median coverage (96%) but more variance.')
print('cyto3 d=200 is most consistent (std=6.3%) with 88% median coverage.')

## 2. Cell Count Distribution

How many cells are detected per image?

This tells us:
- Whether our parameter settings detect a reasonable number of cells
- Whether there are outlier samples (way too many or too few)
- Whether Q and TNF samples tend to have different cell densities

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# ── Left: histogram ───────────────────────────────────────────────────────
ax = axes[0]
q_cells   = df[df['condition'] == 'Quiescent']['n_cells']
tnf_cells = df[df['condition'] == 'TNF']['n_cells']

ax.hist(q_cells,   bins=12, alpha=0.7, color='steelblue', label='Quiescent', edgecolor='white')
ax.hist(tnf_cells, bins=6,  alpha=0.7, color='tomato',    label='TNF',       edgecolor='white')
ax.axvline(df['n_cells'].median(), color='black', linestyle='--', linewidth=1.5,
           label=f'Median = {int(df["n_cells"].median())}')
ax.set_xlabel('Cells detected per image')
ax.set_ylabel('Number of samples')
ax.set_title('Cell Count Distribution (cpsam)')
ax.legend()

# ── Right: bar chart per sample ───────────────────────────────────────────
ax2 = axes[1]
df_sorted = df.sort_values('n_cells', ascending=False).reset_index(drop=True)
bar_c = ['tomato' if c == 'TNF' else 'steelblue' for c in df_sorted['condition']]
ax2.bar(range(len(df_sorted)), df_sorted['n_cells'], color=bar_c, edgecolor='none')
ax2.axhline(df['n_cells'].median(), color='black', linestyle='--', linewidth=1.2)
ax2.set_xlabel('Sample (sorted by cell count)')
ax2.set_ylabel('Cells detected')
ax2.set_title('Cells per Sample')
ax2.legend(handles=[
    mpatches.Patch(color='steelblue', label='Quiescent'),
    mpatches.Patch(color='tomato',    label='TNF'),
    plt.Line2D([0],[0], color='black', linestyle='--', label='Median'),
])

plt.tight_layout()
plt.savefig(EDA_DIR / 'seg_cell_count_distribution.png', bbox_inches='tight')
plt.show()

print('Cell count statistics:')
print(df.groupby('condition')['n_cells'].describe().round(1).to_string())

## 3. Coverage Distribution

**Coverage** = what fraction of the total image area is inside a detected cell mask.

- 100% = every pixel belongs to a cell (impossible in practice — some background always exists)
- 95% = nearly the whole image is cells — good for a dense monolayer
- <70% = significant parts of the image have no detected cells — likely under-segmentation

For endothelial cells growing as a **monolayer**, coverage should be high (80–99%).
Low coverage suggests the model is missing cells or they aren't growing in a monolayer.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# ── Left: histogram ───────────────────────────────────────────────────────
ax = axes[0]
df['coverage_pct'] = df['coverage'] * 100
ax.hist(df[df['condition']=='Quiescent']['coverage_pct'], bins=12,
        alpha=0.7, color='steelblue', label='Quiescent', edgecolor='white')
ax.hist(df[df['condition']=='TNF']['coverage_pct'], bins=6,
        alpha=0.7, color='tomato', label='TNF', edgecolor='white')
ax.axvline(df['coverage_pct'].median(), color='black', linestyle='--', linewidth=1.5,
           label=f'Median = {df["coverage_pct"].median():.1f}%')
ax.set_xlabel('Coverage (%)')
ax.set_ylabel('Number of samples')
ax.set_title('Coverage Distribution (cpsam)')
ax.legend()

# ── Right: scatter — n_cells vs coverage ─────────────────────────────────
ax2 = axes[1]
c_col = ['tomato' if c == 'TNF' else 'steelblue' for c in df['condition']]
ax2.scatter(df['n_cells'], df['coverage_pct'], c=c_col, alpha=0.8, s=60)
ax2.set_xlabel('Number of cells detected')
ax2.set_ylabel('Coverage (%)')
ax2.set_title('Cell Count vs Coverage')
ax2.legend(handles=[
    mpatches.Patch(color='steelblue', label='Quiescent'),
    mpatches.Patch(color='tomato',    label='TNF'),
])

# Annotate outliers
for _, row in df[df['coverage_pct'] < 70].iterrows():
    short = row['sample_name'].split('siractin_')[-1] if 'siractin_' in row['sample_name'] \
            else row['sample_name'][-15:]
    ax2.annotate(short, (row['n_cells'], row['coverage_pct']),
                 fontsize=7, xytext=(5, -10), textcoords='offset points')

plt.tight_layout()
plt.savefig(EDA_DIR / 'seg_coverage_distribution.png', bbox_inches='tight')
plt.show()

print('Coverage statistics:')
print(df.groupby('condition')['coverage_pct'].describe().round(1).to_string())

## 4. Cell Shape and Size — Our CPM Parameters

This is where the thesis gets interesting. We measure 4 morphological properties
per cell. These directly map to Cellular Potts Model (CPM) parameters:

| Measurement | CPM use | What it tells us |
|-------------|---------|------------------|
| **Area** | Target area A_target | How big the cell "wants" to be |
| **Circularity** | Shape energy | How round/irregular the cell boundary is |
| **Eccentricity** | Elongation | How stretched the cell is (0=circle, 1=line) |
| **Solidity** | Compactness | How many concavities the boundary has |

We're looking at the **distributions** of these values across all detected cells.
If Q and TNF cells have different distributions, that's our first evidence for SQ3.

In [ ]:
# Filter out extreme area outliers (likely segmentation artefacts)
# Cells smaller than 100px or larger than 200,000px are almost certainly wrong
cells_clean = cells[(cells['area'] > 100) & (cells['area'] < 200_000)].copy()
print(f'Cells after filtering: {len(cells_clean):,} / {len(cells):,}')

metrics = ['area', 'circularity', 'eccentricity', 'solidity']
labels  = ['Area (px)', 'Circularity', 'Eccentricity', 'Solidity']

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

for ax, metric, label in zip(axes.flat, metrics, labels):
    # Violin plot with condition split
    q_data   = cells_clean[cells_clean['condition'] == 'Quiescent'][metric].dropna()
    tnf_data = cells_clean[cells_clean['condition'] == 'TNF'][metric].dropna()

    parts = ax.violinplot([q_data, tnf_data], positions=[1, 2],
                          showmedians=True, showextrema=True)
    colours = ['steelblue', 'tomato']
    for pc, col in zip(parts['bodies'], colours):
        pc.set_facecolor(col)
        pc.set_alpha(0.6)
    for part_name in ['cmedians', 'cbars', 'cmins', 'cmaxes']:
        if part_name in parts:
            parts[part_name].set_color('black')
            parts[part_name].set_linewidth(1.5)

    ax.set_xticks([1, 2])
    ax.set_xticklabels(['Quiescent\n(n=%d cells)' % len(q_data),
                        'TNF\n(n=%d cells)' % len(tnf_data)])
    ax.set_ylabel(label)
    ax.set_title(label)

    # Annotate medians
    for pos, data, col in zip([1, 2], [q_data, tnf_data], colours):
        med = data.median()
        ax.text(pos, med, f' {med:.2f}', va='center', fontsize=9,
                color='black', fontweight='bold')

fig.suptitle('Cell Morphology: Quiescent vs TNF (cpsam | T0001)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(EDA_DIR / 'seg_morphology_q_vs_tnf.png', bbox_inches='tight')
plt.show()

print('\nMedian values per condition:')
print(cells_clean.groupby('condition')[metrics].median().round(3).to_string())

## 5. RhoA vs RhoB: Morphological Differences?

We also have two different Rho GTPase proteins — **RhoA** and **RhoB**.
Both belong to the same family but have different roles:
- **RhoA**: primarily regulates actomyosin contractility (cell tension)
- **RhoB**: involved in vesicle trafficking and stress response

Do cells expressing RhoA vs RhoB look morphologically different?
This is an exploratory check — not a primary hypothesis.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

for ax, metric, label in zip(axes.flat, metrics, labels):
    rhoa_data = cells_clean[cells_clean['protein'] == 'RhoA'][metric].dropna()
    rhob_data = cells_clean[cells_clean['protein'] == 'RhoB'][metric].dropna()

    parts = ax.violinplot([rhoa_data, rhob_data], positions=[1, 2],
                          showmedians=True, showextrema=True)
    colours = ['#5B9BD5', '#ED7D31']
    for pc, col in zip(parts['bodies'], colours):
        pc.set_facecolor(col)
        pc.set_alpha(0.6)
    for part_name in ['cmedians', 'cbars', 'cmins', 'cmaxes']:
        if part_name in parts:
            parts[part_name].set_color('black')

    ax.set_xticks([1, 2])
    ax.set_xticklabels(['RhoA\n(n=%d cells)' % len(rhoa_data),
                        'RhoB\n(n=%d cells)' % len(rhob_data)])
    ax.set_ylabel(label)
    ax.set_title(label)
    for pos, data, col in zip([1, 2], [rhoa_data, rhob_data], colours):
        med = data.median()
        ax.text(pos, med, f' {med:.2f}', va='center', fontsize=9, fontweight='bold')

fig.suptitle('Cell Morphology: RhoA vs RhoB (cpsam | T0001)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(EDA_DIR / 'seg_morphology_rhoa_vs_rhob.png', bbox_inches='tight')
plt.show()

print('\nMedian values per protein:')
print(cells_clean.groupby('protein')[metrics].median().round(3).to_string())

## 6. Model Comparison: cpsam vs cyto3

We tested two types of deep learning models:

- **cpsam (Transformer)**: Uses a "vision transformer" — the same type of architecture
  as ChatGPT but for images. Very powerful, 312 million parameters, 1.15 GB file.
  Like hiring a very smart scientist who takes ~36 seconds per image.

- **cyto3 (CNN)**: Uses a convolutional neural network — the classic image recognition
  architecture. 25 MB file. Like hiring an experienced technician who takes ~24 seconds.

We tested cyto3 with different **diameter** settings (150px vs 200px):
- Diameter tells the model how big a typical cell is
- Getting this wrong causes over-segmentation (too many tiny pieces) or under-segmentation

And with different **channel combinations** (Actin alone vs Actin + VE-cadherin).

In [ ]:
# Prepare comparison data
comp_rows = []
for label, s_df in summaries.items():
    model_type = 'Transformer\n(cpsam)' if 'cpsam' in label else 'CNN\n(cyto3)'
    comp_rows.append({
        'label': label,
        'model_type': model_type,
        'median_cov': s_df['coverage'].median() * 100,
        'mean_cov': s_df['coverage'].mean() * 100,
        'std_cov': s_df['coverage'].std() * 100,
        'median_cells': s_df['n_cells'].median(),
    })
comp_df = pd.DataFrame(comp_rows)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Left: median coverage ─────────────────────────────────────────────────
ax = axes[0]
bar_colours = ['#2196F3' if 'cpsam' in r['label'] else '#FF9800'
               for _, r in comp_df.iterrows()]
bars = ax.bar(range(len(comp_df)), comp_df['median_cov'],
              color=bar_colours, edgecolor='white')
ax.errorbar(range(len(comp_df)), comp_df['median_cov'],
            yerr=comp_df['std_cov'], fmt='none', color='black',
            capsize=5, linewidth=1.5)
for i, (bar, row) in enumerate(zip(bars, comp_df.itertuples())):
    ax.text(i, row.median_cov + row.std_cov + 0.5,
            f'{row.median_cov:.0f}%', ha='center', fontsize=9, fontweight='bold')
ax.set_xticks(range(len(comp_df)))
ax.set_xticklabels(
    [r.replace(' (', '\n(') for r in comp_df['label']],
    fontsize=8, rotation=15, ha='right'
)
ax.set_ylabel('Median coverage (%)')
ax.set_title('Segmentation Coverage\nby Model & Configuration')
ax.set_ylim(50, 105)
ax.legend(handles=[
    mpatches.Patch(color='#2196F3', label='cpsam (Transformer)'),
    mpatches.Patch(color='#FF9800', label='cyto3 (CNN)'),
])

# ── Right: consistency (std coverage) ────────────────────────────────────
ax2 = axes[1]
ax2.bar(range(len(comp_df)), comp_df['std_cov'],
        color=bar_colours, edgecolor='white')
for i, v in enumerate(comp_df['std_cov']):
    ax2.text(i, v + 0.2, f'{v:.1f}%', ha='center', fontsize=9)
ax2.set_xticks(range(len(comp_df)))
ax2.set_xticklabels(
    [r.replace(' (', '\n(') for r in comp_df['label']],
    fontsize=8, rotation=15, ha='right'
)
ax2.set_ylabel('Std coverage (%) — lower is more consistent')
ax2.set_title('Segmentation Consistency\n(Standard Deviation of Coverage)')

plt.suptitle('Model Comparison: cpsam (Transformer) vs cyto3 (CNN)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(EDA_DIR / 'seg_model_comparison.png', bbox_inches='tight')
plt.show()

print('\nFull comparison table:')
print(comp_df[['label', 'median_cov', 'mean_cov', 'std_cov', 'median_cells']]
      .round(1).to_string(index=False))

## 7. Feature Correlations — Are the Metrics Related?

Before using cell shape metrics as independent features, we should check:
**do bigger cells also tend to be rounder? Do elongated cells tend to be less solid?**

If two metrics are perfectly correlated (r ≈ 1.0), they're measuring the same thing
and we only need one. If they're independent (r ≈ 0), they each add unique information.

> A **correlation heatmap** shows all pairwise correlations at once.
> Red = positive correlation (both go up together).
> Blue = negative correlation (one goes up, the other goes down).
> White/light = no correlation.

In [ ]:
feature_cols = ['area', 'perimeter', 'circularity', 'eccentricity', 'solidity', 'mean_intensity']
corr_matrix  = cells_clean[feature_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.zeros_like(corr_matrix, dtype=bool)
mask[np.triu_indices_from(mask, k=1)] = True  # hide upper triangle (it's a mirror)

sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.5,
    ax=ax, annot_kws={'fontsize': 10}
)
ax.set_title('Feature Correlation Matrix (all detected cells, cpsam)',
             fontweight='bold')
plt.tight_layout()
plt.savefig(EDA_DIR / 'seg_feature_correlation_heatmap.png', bbox_inches='tight')
plt.show()

# Highlight notable correlations
print('Notable correlations (|r| > 0.4):')
for i, c1 in enumerate(feature_cols):
    for j, c2 in enumerate(feature_cols):
        if j >= i:
            continue
        r = corr_matrix.loc[c1, c2]
        if abs(r) > 0.4:
            direction = 'positive' if r > 0 else 'negative'
            print(f'  {c1} × {c2}: r={r:.2f} ({direction})')

## 8. Outlier Analysis

Three samples produced unexpectedly extreme results.
Let's identify and investigate them.

**Why this matters:**
Outliers can pull group averages in misleading directions.
By understanding *why* they're outliers, we can decide whether to:
- Include them (if they're biologically valid)
- Flag them in the analysis (add a caveat)
- Exclude them (only if there's a clear technical reason)

In [ ]:
# Define outlier criteria
med_cov    = df['coverage_pct'].median()
med_cells  = df['n_cells'].median()
std_cov    = df['coverage_pct'].std()
std_cells  = df['n_cells'].std()

# Flag samples more than 1.5 std from the median
df['cov_z']   = (df['coverage_pct']  - med_cov)   / std_cov
df['cells_z'] = (df['n_cells'] - med_cells) / std_cells
outliers = df[abs(df['cov_z']) > 1.5].copy()

print(f'Outlier samples (coverage more than 1.5 std from median):' )
print(f'  Median coverage: {med_cov:.1f}%, Std: {std_cov:.1f}%')
print()
if len(outliers) > 0:
    for _, row in outliers.sort_values('cov_z').iterrows():
        direction = 'LOW' if row['cov_z'] < 0 else 'HIGH'
        short = row['sample_name'].split('siractin_')[-1] if 'siractin_' in row['sample_name'] \
                else row['sample_name'][-30:]
        print(f'  [{direction}] {short}')
        print(f'    Condition: {row["condition"]} | Protein: {row["protein"]}')
        print(f'    Coverage: {row["coverage_pct"]:.1f}% (z={row["cov_z"]:.2f}) | Cells: {row["n_cells"]}')
        if direction == 'LOW':
            print('    → Likely cpsam over-segmented (many tiny fragments = high cell count, low coverage)')
        else:
            print('    → Unusually high coverage — may indicate fused cell masks')
        print()
else:
    print('  No strong outliers detected.')

# Visualise outliers in context
fig, ax = plt.subplots(figsize=(10, 4))
c_col = ['tomato' if c == 'TNF' else 'steelblue' for c in df['condition']]
ax.scatter(df['n_cells'], df['coverage_pct'], c=c_col, alpha=0.7, s=70, zorder=3)

# Highlight outliers
ax.scatter(outliers['n_cells'], outliers['coverage_pct'],
           c='none', s=200, edgecolors='red', linewidths=2, zorder=4, label='Outlier')
for _, row in outliers.iterrows():
    short = row['sample_name'].split('siractin_')[-1] if 'siractin_' in row['sample_name'] \
            else row['sample_name'][-18:]
    ax.annotate(short, (row['n_cells'], row['coverage_pct']),
                fontsize=7, xytext=(5, 5), textcoords='offset points')

# Confidence zone
ax.axhspan(med_cov - 1.5*std_cov, med_cov + 1.5*std_cov,
           alpha=0.08, color='green', label='Within 1.5 std')
ax.set_xlabel('Number of cells detected')
ax.set_ylabel('Coverage (%)')
ax.set_title('Outlier Samples: Coverage vs Cell Count (cpsam)')
ax.legend(handles=[
    mpatches.Patch(color='steelblue', label='Quiescent'),
    mpatches.Patch(color='tomato', label='TNF'),
    plt.scatter([], [], c='none', edgecolors='red', s=100, linewidths=2, label='Outlier'),
])

plt.tight_layout()
plt.savefig(EDA_DIR / 'seg_outlier_analysis.png', bbox_inches='tight')
plt.show()


## 9. Visual QC: Raw Image + Segmentation Mask

Quantitative outlier detection tells us *which* samples are problematic.
But to understand *why* — image quality issue, or model failure on a normal-looking
image — we need to look at the actual images and their segmentation masks side by side.

**This is standard practice in medical image EDA:**  
Always visually inspect flagged samples before deciding to exclude or keep them.

**What to look for per sample:**
- **Good sample:** cells clearly visible in raw image; mask boundaries align with real cell edges
- **Outlier (image quality):** raw image looks blurry or very dim → model had no information
- **Outlier (model failure):** raw image looks fine but mask is fragmented → parameter issue

Each row shows one sample. Three panels:
1. **Raw Actin (Ch3)** — the actual input fed to Cellpose (contrast-stretched for display)
2. **Coloured mask** — each detected cell gets a unique colour; lets you see coverage and count
3. **Outline overlay** — cell boundaries drawn on the raw image; best for checking alignment


In [ ]:

from PIL import Image as _PIL_Image

# ── Paths ──────────────────────────────────────────────────────────────────
_DATA_DIR  = PROJECT_ROOT / 'data' / 'JK2513_A1R_HUVEC_dataset cellular pots model'
_MASKS_DIR = OUTPUT_DIR / 'batch_t0001_cpsam_tiff' / 'samples'

def _load_tiff_vis(path):
    return np.array(_PIL_Image.open(path))

def _display_norm(img, pct=99.0):
    lo, hi = img.min(), np.percentile(img, pct)
    if hi == lo:
        return np.zeros_like(img, dtype=float)
    return np.clip((img.astype(float) - lo) / (hi - lo), 0, 1)

def _find_actin_t0001(sample_name, data_dir):
    """Locate the T0001 Actin TIFF for a given sample_name."""
    for ch3_dir in data_dir.rglob('*_Channel3'):
        if 'Out of focus' in str(ch3_dir):
            continue
        dir_s = ch3_dir.name.replace('_Channel3', '')
        if dir_s == sample_name or sample_name.endswith(dir_s):
            hits = sorted(ch3_dir.glob('*T0001*.tif'))
            if hits:
                return hits[0]
    return None

# ── Select samples: 2 best (highest coverage) + all outliers ──────────────
_df2 = df.copy()
if 'coverage_pct' not in _df2.columns:
    _df2['coverage_pct'] = _df2['coverage'] * 100
_med2 = _df2['coverage_pct'].median()
_std2 = _df2['coverage_pct'].std()
_df2['_z2'] = (_df2['coverage_pct'] - _med2) / _std2

_good_rows     = _df2.nlargest(2, 'coverage_pct')
_outlier_rows  = _df2[abs(_df2['_z2']) > 1.5].sort_values('_z2')
_show          = pd.concat([_good_rows, _outlier_rows]).drop_duplicates('sample_name').head(6)
n_show         = len(_show)

# ── Figure ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(n_show, 3, figsize=(15, 4 * n_show))
if n_show == 1:
    axes = axes[np.newaxis, :]

for ci, ctitle in enumerate(['Raw Actin (Ch3)', 'Coloured Mask', 'Outline Overlay']):
    axes[0, ci].set_title(ctitle, fontsize=11, fontweight='bold')

for ri, (_, row) in enumerate(_show.iterrows()):
    sname   = row['sample_name']
    cov_val = row['coverage_pct']
    z_val   = row['_z2']
    is_out  = abs(z_val) > 1.5
    tag_col = 'tomato' if is_out else 'steelblue'
    tag_str = 'OUTLIER' if is_out else 'GOOD'
    short   = sname.split('siractin_')[-1] if 'siractin_' in sname else sname[-22:]

    # Locate raw Actin TIFF
    actin_p = _find_actin_t0001(sname, _DATA_DIR)

    # Locate mask and outline PNGs (exact name → fuzzy fallback)
    mask_p    = _MASKS_DIR / f'{sname}_masks.png'
    outline_p = _MASKS_DIR / f'{sname}_outlines.png'
    if not mask_p.exists():
        cands = sorted(_MASKS_DIR.glob(f'*{sname[-12:]}*masks*'))
        mask_p = cands[0] if cands else None
    if not outline_p.exists():
        cands = sorted(_MASKS_DIR.glob(f'*{sname[-12:]}*outlines*'))
        outline_p = cands[0] if cands else None

    row_label = (f'[{tag_str}]  {short}\n'
                 f'{row["condition"]} | {int(row["n_cells"])} cells | {cov_val:.0f}% cov')

    # Panel 1: raw image
    if actin_p and actin_p.exists():
        raw = _load_tiff_vis(actin_p)
        axes[ri, 0].imshow(_display_norm(raw), cmap='gray')
    else:
        axes[ri, 0].text(0.5, 0.5, 'File not found', ha='center', va='center',
                          transform=axes[ri, 0].transAxes, color='gray', fontsize=9)
    axes[ri, 0].axis('off')
    axes[ri, 0].set_ylabel(row_label, fontsize=8, color=tag_col, rotation=0,
                            ha='right', va='center', labelpad=5)

    # Panel 2: coloured mask overlay
    if mask_p and Path(str(mask_p)).exists():
        axes[ri, 1].imshow(np.array(_PIL_Image.open(str(mask_p))))
    else:
        axes[ri, 1].text(0.5, 0.5, 'Not found', ha='center', va='center',
                          transform=axes[ri, 1].transAxes, color='gray')
    axes[ri, 1].axis('off')

    # Panel 3: outline overlay
    if outline_p and Path(str(outline_p)).exists():
        axes[ri, 2].imshow(np.array(_PIL_Image.open(str(outline_p))))
    else:
        axes[ri, 2].text(0.5, 0.5, 'Not found', ha='center', va='center',
                          transform=axes[ri, 2].transAxes, color='gray')
    axes[ri, 2].axis('off')

fig.suptitle(
    'Visual QC: Raw Actin Image vs Segmentation Overlay\n'
    'Good samples (top, blue) vs Outlier samples (bottom, red) — cpsam | T0001',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.savefig(EDA_DIR / 'seg_visual_qc_image_mask.png', bbox_inches='tight', dpi=130)
plt.show()
print(f'Saved → {EDA_DIR / "seg_visual_qc_image_mask.png"}')



## 10. External Reference Datasets

EDA best practice — and the UvA thesis requirements — ask us to compare with external
datasets to: (1) sanity-check our cell measurements, and (2) situate the work in context.

### Why this is hard for our dataset

Our setup is quite niche:
- **HUVEC primary endothelial cells** — not in any major public benchmark
- **Fluorescence, 3-channel, 16-bit TIFF** — most benchmarks use phase-contrast or 8-bit
- **Time-lapse for CPM parameterization** — no existing benchmark addresses this task
- **No ground-truth annotations** — can't compute standard IoU; use coverage as proxy

We compare with **4 external datasets**, ordered from most to least relevant to our thesis:

| Priority | Dataset | Why relevant |
|----------|---------|--------------|
| ★★★★★ | ECPT / King's College HUVEC | Same cell type (HUVEC) |
| ★★★★ | Cell Tracking Challenge | Time-lapse + tracking ground truth |
| ★★★ | LIVECell (Edlund 2021) | Monolayer cells + Cellpose benchmark |
| ★★ | DSB 2018 (Kaggle) | Fluorescence microscopy, widely cited |


In [ ]:

# ── Our dataset snapshot ───────────────────────────────────────────────────
_our_cov   = df['coverage_pct'].median() if 'coverage_pct' in df.columns else df['coverage'].median() * 100
_our_cells = int(df['n_cells'].median())
_our_area  = int(cells_clean['area'].median()) if len(cells_clean) > 0 else 'N/A'

print('─' * 72)
print('OUR DATASET — Snapshot')
print('─' * 72)
print(f'  23 HUVEC samples | 1,261 frames | cpsam | Actin Ch3 | 16-bit TIFF')
print(f'  Median cells / image : {_our_cells}')
print(f'  Median coverage      : {_our_cov:.1f}% (proxy; no ground-truth annotations)')
print(f'  Median cell area     : {_our_area:,} px (at 1024×1024 resolution)')
print()

# ── Structured comparison table ────────────────────────────────────────────
ext_data = {
    'Dataset': [
        'This thesis (HUVEC)',
        "ECPT / King's College",
        'Cell Tracking Challenge',
        'LIVECell (Edlund 2021)',
        'DSB 2018 (Kaggle)',
    ],
    'Cell type': [
        'HUVEC (primary endothelial)',
        'HUVEC (primary endothelial)',
        'Various incl. endothelial',
        '8 cancer cell lines (monolayer)',
        'Nuclei (diverse cell types)',
    ],
    'Microscopy': [
        'Fluorescence 16-bit TIFF',
        'Confocal fluorescence',
        'Phase-contrast / fluorescence',
        'Phase-contrast 8-bit',
        'Fluorescence + brightfield',
    ],
    'Time-lapse?': ['Yes (1,261 frames)', 'No', 'Yes', 'No', 'No'],
    'Ground truth?': ['No (proxy only)', 'Semi (CellProfiler)', 'Yes (SEG+TRA)', 'Yes (1.6M cells)', 'Yes (nuclei)'],
    'Cellpose used?': ['Yes (cpsam)', 'No', 'No', 'Yes (benchmark)', 'Rarely'],
    'Relevance': ['—', '★★★★★ Same cells', '★★★★ Time-lapse+GT', '★★★ Monolayer+Cellpose', '★★ Fluorescence'],
}
ext_df = pd.DataFrame(ext_data)
print('External Dataset Comparison:')
print(ext_df.to_string(index=False))
print()

# ── LIVECell: detailed similarities vs differences ─────────────────────────
print('─' * 72)
print('LIVECELL — Similarities and Differences')
print('Edlund et al. 2021, Nature Methods | doi:10.1038/s41592-021-01249-6')
print('─' * 72)

print('\nSIMILARITIES (why LIVECell is our primary sanity-check):')
for sim in [
    'Both feature adherent monolayer cells (single layer, cells touching each other)',
    'Both use Cellpose — directly comparable model characteristics and behaviour',
    'Similar cell count per frame: LIVECell 10–80 cells; ours median 29 cells',
    'Dense culture → high expected coverage in both (cells fill most of the image)',
    'Both 2D (single focal plane) → same segmentation approach is applicable',
]:
    print(f'  ✓ {sim}')

print('\nDIFFERENCES (why LIVECell numbers cannot be directly applied):')
diffs = [
    ('Cell type',
     'LIVECell: cancer cell lines (A172, Huh7, MCF7, BT474, SKOV3, SHSY5Y, BV2, SkBr3)',
     'Ours: HUVEC — primary human vascular endothelial cells. Very different biology,'
     ' cell size, and adhesion properties.'),
    ('Microscopy',
     'LIVECell: phase-contrast (bright background, dark cell outlines — inverted contrast)',
     'Ours: fluorescence (dark background, bright cell bodies from actin staining).'
     ' Completely different pixel intensity distribution.'),
    ('Bit depth',
     'LIVECell: 8-bit grayscale (256 intensity levels)',
     'Ours: 16-bit TIFF with 12-bit sensor range (4,096 intensity levels).'
     ' 16× richer dynamic range — key for preserving fine intensity gradients.'),
    ('Ground truth',
     'LIVECell: ~1.6 million expert-annotated cell boundaries → IoU computed directly',
     'Ours: no annotations → coverage (fraction of image inside detected cells) as proxy.'
     ' Cannot compute IoU without ground truth.'),
    ('Time dimension',
     'LIVECell: static single frames — no dynamics, no tracking',
     'Ours: time-lapse videos (19–95 frames / sample). Cell shape changes over time.'
     ' Tracking and temporal features are central to our CPM parameterization goal.'),
    ('Purpose',
     'LIVECell: general segmentation benchmark (measure IoU against human annotations)',
     'Ours: CPM parameter extraction and Q vs TNF biological comparison.'
     ' Segmentation is a means to an end, not the final goal.'),
    ('Cellpose IoU',
     'LIVECell: Cellpose cyto2 achieves IoU 0.54–0.66 (varies by cell line)',
     'Ours: cpsam achieves 96% median coverage. Consistent with strong performance'
     ' on dense monolayers, but not directly comparable without ground truth.'),
]
for label, livecell_val, our_val in diffs:
    print(f'\n  ✗ {label}:')
    print(f'      LIVECell : {livecell_val}')
    print(f'      Ours     : {our_val}')

print()
print('─' * 72)
print('CONCLUSION: Why no existing benchmark directly applies to our task')
print('─' * 72)
print('''
  The closest match by cell type (ECPT/King's College HUVEC) uses a classical
  CellProfiler pipeline, no time-lapse, and no deep learning — so our DL comparison
  cannot be validated against it.

  The closest match by task (Cell Tracking Challenge) provides time-lapse + tracking
  ground truth for endothelial cells but only in phase-contrast, not fluorescence.

  LIVECell confirms that Cellpose performs well (IoU 0.54–0.66) on dense monolayer
  cells. Our 96% median coverage is consistent with strong performance, but measured
  differently (coverage proxy vs IoU with ground truth).

  DSB 2018 is fluorescence but targets nuclei — a different, easier segmentation target
  than the full cell body we segment from actin staining.

  This gap — fluorescence HUVEC time-lapse segmentation for CPM parameterization —
  is the novel contribution of this thesis and the reason existing benchmarks
  cannot be applied directly to evaluate our pipeline.
''')


## Key Takeaways and Decisions for the Thesis

### What we found:

| Finding | Detail |
|---------|--------|
| **cpsam = best coverage** | Median 96% vs 88% for best cyto3 config |
| **cyto3 = more consistent** | Std 6% vs 14.5% for cpsam — fewer extreme failures |
| **3 outlier samples** | cpsam over-segments `26112025_Q_002`, `15012026_Q_001`, `15012026_TNF_001` |
| **Actin-only > multichannel** | Adding VE-cadherin channel *hurts* — reduces coverage 84%→78% |
| **Q vs TNF morphology** | Visual difference detectable; formal test deferred to SQ3 |
| **Features are not fully independent** | Area and perimeter strongly correlated; may need to reduce features |

### Decisions for the thesis pipeline:

1. **Primary segmentation model**: `cpsam` — highest coverage, best for per-frame analysis
2. **Channel**: Ch3 (Actin) only — adding Ch1 (VE-cadherin) consistently hurts
3. **Outlier samples**: Flag but do not exclude — include in full analysis with caveat
4. **Features to extract**: area, circularity, eccentricity, solidity → 4 morphological CPM parameters
5. **Next step**: Run segmentation on ALL 1,261 frames (not just T0001) to enable tracking

---
*EDA complete. Next milestone: Full time-series segmentation → cell tracking → feature extraction.*